In [ ]:
import cv2
import mediapipe as mp
import time

# ==============================================================================
# 1. MediaPipe Tasks API 핵심 클래스 로드
# ==============================================================================
# BaseOptions: 모델 파일 경로(.task) 및 디바이스(CPU/GPU) 설정 담당
BaseOptions = mp.tasks.BaseOptions

# HandLandmarker: 실제 손 관절 위치를 추론하는 메인 클래스
HandLandmarker = mp.tasks.vision.HandLandmarker

# HandLandmarkerOptions: 신뢰도 임계값, 인식 모드, 최대 손 개수 등을 세팅하는 옵션 클래스
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

# VisionRunningMode: 입력 데이터 형태 정의 (IMAGE: 정지영상, VIDEO: 비디오 파일, LIVE_STREAM: 실시간 웹캠)
VisionRunningMode = mp.tasks.vision.RunningMode


# ==============================================================================
# 2. 손 관절(Landmark) 연결선 정의 (시각화용)
# ==============================================================================
# 21개 관절 점 인덱스를 기반으로 손가락 마디마디를 잇는 튜플 리스트
HAND_CONNECTIONS = [
    # 엄지손가락 (Wrist 0 -> 손목 base 1 -> 관절 2, 3 -> 손끝 4)
    (0, 1), (1, 2), (2, 3), (3, 4),
    
    # 검지손가락 (Wrist 0 -> 검지 뿌리 5 -> 관절 6, 7 -> 손끝 8)
    (0, 5), (5, 6), (6, 7), (7, 8),
    
    # 중지손가락 (검지 뿌리 5 -> 중지 뿌리 9 -> 관절 10, 11 -> 손끝 12)
    (5, 9), (9, 10), (10, 11), (11, 12),
    
    # 약지손가락 (중지 뿌리 9 -> 약지 뿌리 13 -> 관절 14, 15 -> 손끝 16)
    (9, 13), (13, 14), (14, 15), (15, 16),
    
    # 새끼손가락 (약지 뿌리 13 -> 새끼 뿌리 17 -> 관절 18, 19 -> 손끝 20 + 손목 0 연결)
    (13, 17), (0, 17), (17, 18), (18, 19), (19, 20)
]


# ==============================================================================
# 3. HandLandmarker 옵션 설정 및 모델 로드
# ==============================================================================
options = HandLandmarkerOptions(
    # 다운로드받은 모델 파일(.task)의 상대/절대 경로 지정
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    
    # 비디오 스트림 처리 모드로 설정 (동적 타임스탬프와 함께 연속 프레임 추적)
    running_mode=VisionRunningMode.VIDEO,
    
    # 화면 내 감지할 최대 손 개수 설정 (1로 설정 시 가장 명확한 손 1개만 추적)
    num_hands=1,
    
    # 손 탐지(Detection) 최소 신뢰도 (0.5 = 50% 이상 확신할 때 손으로 인식)
    min_hand_detection_confidence=0.5,
    
    # 손 추적(Tracking) 최소 신뢰도 (프레임 간 손의 위치를 계속 추적하는 임계값)
    min_tracking_confidence=0.5
)

# 기본 연결된 웹캠 장치 열기 (0: 기본 카메라)
cap = cv2.VideoCapture(0)


# ==============================================================================
# 4. 실시간 추론 및 시각화 루프
# ==============================================================================
# 'with' 문을 통해 Landmarker 객체를 안전하게 생성 (작업 종료 시 메모리 자원 자동 해제)
with HandLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        # 웹캠으로부터 1개 프레임 읽어오기 (ret: 성공여부 Boolean, frame: BGR 이미지 배열)
        ret, frame = cap.read()
        if not ret:
            print("카메라 프레임을 읽어올 수 없습니다.")
            break

        # 거울 모드 처리를 위한 좌우 반전 (1: 좌우 반전)
        frame = cv2.flip(frame, 1)
        
        # OpenCV의 BGR 색상 채널을 MediaPipe가 지원하는 RGB 채널로 변환
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # NumPy 배열 이미지를 MediaPipe의 전용 Image 객체로 래핑
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        # Tasks API VIDEO 모드 필수값: 밀리초(ms) 단위의 단조 증가 타임스탬프 생성
        frame_timestamp_ms = int(time.time() * 1000)
        
        # RGB 이미지와 타임스탬프를 모델에 전달하여 3D 관절 추론 실행
        result = landmarker.detect_for_video(mp_image, frame_timestamp_ms)

        # 원본 프레임의 높이(h), 너비(w) 추출 (정규화 좌표 -> 픽셀 좌표 변환용)
        h, w, _ = frame.shape
        
        # 손이 화면에서 성공적으로 감지되었는지 확인
        if result.hand_landmarks:
            # 감지된 손 개수만큼 반복 (num_hands=1 설정으로 최대 1회 실행)
            for hand_landmarks in result.hand_landmarks:
                pixel_coords = [] # 21개 관절의 (x, y) 픽셀 좌표를 담을 리스트
                
                # --------------------------------------------------------------
                # 4-1. 관절 점(Landmark) 위치 계산 및 출력
                # --------------------------------------------------------------
                for lm in hand_landmarks:



                    # lm.x, lm.y는 0.0~1.0 사이로 정규화된 값이므로 해상도(w, h)를 곱해 픽셀 단위로 변환
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    pixel_coords.append((cx, cy))
                    
                    # 관절 위치에 녹색 원 그리기 (원 중심: cx,cy / 반지름: 4px / 색상: BGR (0,255,0) / 채우기: -1)
                    cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)

                # --------------------------------------------------------------
                # 4-2. 관절 마디 연결선 그리기
                # --------------------------------------------------------------
                for start_idx, end_idx in HAND_CONNECTIONS:
                    # HAND_CONNECTIONS 상의 두 관절 픽셀 좌표 획득
                    p1 = pixel_coords[start_idx]
                    p2 = pixel_coords[end_idx]
                    
                    # 두 점 사이에 파란색 직선 그리기 (색상: BGR (255,0,0) / 두께: 2px)
                    cv2.line(frame, p1, p2, (255, 0, 0), 2)

        # 처리 결과 프레임을 "Hand Recognition" 윈도우 창에 표시
        cv2.imshow("Hand Recognition", frame)

        # 5밀리초 동안 키 입력을 대기하고 8비트 마스크 적용
        key = cv2.waitKey(5) & 0xFF
        # ESC 키(ASCII 27) 또는 'q' 키를 누르면 루프 종료
        if key == 27 or key == ord('q'):
            break

# ==============================================================================
# 5. 자원 해제
# ==============================================================================
# 웹캠 장치 점유 해제
cap.release()

# 오픈되어 있는 모든 OpenCV 그래픽 창 닫기
cv2.destroyAllWindows()

#### 기본 컴퓨터 가위바위보

In [3]:
import random

# 선택지 및 매핑
choices = {0: "가위", 1: "바위", 2: "보"}

print("=== 가위바위보 게임 ===")
user_input = int(input("가위(0), 바위(1), 보(2) 중 하나를 선택하세요: "))

if user_input not in [0, 1, 2]:
    print("잘못된 입력입니다.")
else:
    # 1. 컴퓨터의 무작위 선택
    computer_choice = random.randint(0, 2)
    
    print(f"사용자: {choices[user_input]} vs 컴퓨터: {choices[computer_choice]}")
    
    # 2. 승패 판정 (모듈로 연산 활용)
    result = (user_input - computer_choice) % 3
    
    if result == 0:
        print("결과: 비겼습니다!")
    elif result == 1:
        print("결과: 이겼습니다!")
    else:
        print("결과: 졌습니다!")

=== 가위바위보 게임 ===
사용자: 바위 vs 컴퓨터: 보
결과: 졌습니다!


#### Hand Landmark 손가위, 바위, 보 게임

In [ ]:
# OpenCV 라이브러리: 비디오 프레임 처리 및 화면 출력을 담당
import cv2

# MediaPipe 라이브러리: 손 관절(Landmark) 추론 및 컴퓨터 비전 기능 제공
import mediapipe as mp

# random 라이브러리: 컴퓨터의 가위/바위/보 무작위 선택에 사용
import random

# time 라이브러리: MediaPipe Tasks API 입력에 필요한 밀리초 타임스탬프 계산
import time

# ==============================================================================
# 1. MediaPipe Tasks API 필수 모듈 및 설정 로드
# ==============================================================================
# BaseOptions: 모델 파일 경로(.task) 및 CPU/GPU 디바이스 지정 옵션
BaseOptions = mp.tasks.BaseOptions

# HandLandmarker: 21개 손 관절 좌표를 추론하는 메인 클래스
HandLandmarker = mp.tasks.vision.HandLandmarker

# HandLandmarkerOptions: 추론 신뢰도, 감지할 손 개수, 실행 모드를 세팅하는 클래스
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

# VisionRunningMode: 입력 형태 설정 (IMAGE: 이미지, VIDEO: 비디오 파일, LIVE_STREAM: 비동기 스트림)
VisionRunningMode = mp.tasks.vision.RunningMode

# 손 관절 뼈대를 이어줄 21개 랜드마크 인덱스 쌍 정의 (시각화 목적)
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),        # 엄지손가락 마디 연결선
    (0, 5), (5, 6), (6, 7), (7, 8),        # 검지손가락 마디 연결선
    (5, 9), (9, 10), (10, 11), (11, 12),   # 중지손가락 마디 연결선
    (9, 13), (13, 14), (14, 15), (15, 16), # 약지손가락 마디 연결선
    (13, 17), (0, 17), (17, 18), (18, 19), (19, 20) # 새끼손가락 마디 연결선
]

# ==============================================================================
# 2. 손지형(가위, 바위, 보) 판별 함수
# ==============================================================================
# 21개 관절 좌표(landmarks)를 전달받아 가위/바위/보 또는 알 수 없음 문자열을 반환하는 함수
def classify_rps(landmarks):
    # 검지 뿌리(5)와 새끼 뿌리(17)의 x좌표 위치를 비교하여 손바닥이 바라보는 방향 판별
    # 검지 뿌리가 오른쪽에 있으면 오른손, 왼쪽에 있으면 왼손 기준 적용
    if landmarks[5].x > landmarks[17].x:
        # 오른손 기준: 엄지 끝(4)의 x좌표가 엄지 마디(3)보다 오른쪽(더 큼)에 있으면 펴진 상태
        thumb_open = landmarks[4].x > landmarks[3].x
    else:
        # 왼손 기준: 엄지 끝(4)의 x좌표가 엄지 마디(3)보다 왼쪽(더 작음)에 있으면 펴진 상태
        thumb_open = landmarks[4].x < landmarks[3].x

    # 나머지 4개 손가락은 y축 좌표 비교 (화면 상단이 0, 하단이 1이므로 y값이 작을수록 위로 펴짐)
    # 검지 손가락 끝(8)이 중간 마디(6)보다 위(y값이 작음)에 있는지 확인
    index_open = landmarks[8].y < landmarks[6].y
    
    # 중지 손가락 끝(12)이 중간 마디(10)보다 위에 있는지 확인
    middle_open = landmarks[12].y < landmarks[10].y
    
    # 약지 손가락 끝(16)이 중간 마디(14)보다 위에 있는지 확인
    ring_open = landmarks[16].y < landmarks[14].y
    
    # 새끼 손가락 끝(20)이 중간 마디(18)보다 위에 있는지 확인
    pinky_open = landmarks[20].y < landmarks[18].y

    # 가위 조건 1: (엄지+검지)가 펴지고 중지, 약지, 새끼는 굽혀진 경우 (권총 모양)
    scissors_thumb_index = thumb_open and index_open and (not middle_open) and (not ring_open) and (not pinky_open)
    
    # 가위 조건 2: (검지+중지)가 펴지고 엄지, 약지, 새끼는 굽혀진 경우 (V 자 모양)
    scissors_index_middle = (not thumb_open) and index_open and middle_open and (not ring_open) and (not pinky_open)

    # 가위 조건 3: (약지 + 소지)가 펴지고 엄지, 검지, 중지는 굽혀진 경우 (작은 v 자 모양)
    scissors_index_middle2 = (not thumb_open) and (not index_open) and (not middle_open) and ring_open and pinky_open

    # 가위 조건 1 또는 2 중 하나라도 만족 시 "가위" 반환
    if scissors_thumb_index or scissors_index_middle:
        return "가위"
    
    # 보 조건: 엄지를 포함한 5개 손가락이 모두 펴진 경우 "보" 반환
    elif thumb_open and index_open and middle_open and ring_open and pinky_open:
        return "보"
    
    # 바위 조건: 엄지를 포함한 5개 손가락이 모두 굽혀진 경우 "바위" 반환
    elif (not thumb_open) and (not index_open) and (not middle_open) and (not ring_open) and (not pinky_open):
        return "바위"
    
    # 위 조건들에 해당하지 않는 부정확한 손짓일 경우 "알 수 없음" 반환
    else:
        return "알 수 없음"

# ==============================================================================
# 3. 메인 가위바위보 실행 및 UI 설정
# ==============================================================================
# MediaPipe HandLandmarker 모델 파라미터 및 인스턴스 옵션 구성
options = HandLandmarkerOptions(
    # 로컬 경로에 저장된 손 인식 모델 파일(.task) 지정
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    # 웹캠 비디오 프레임 스트림 처리를 위한 VIDEO 모드 선택
    running_mode=VisionRunningMode.VIDEO,
    # 추론할 최대 손 개수 (1개로 제한하여 오작동 방지)
    num_hands=1,
    # 최초 손 탐지 신뢰도 임계값 (50% 이상)
    min_hand_detection_confidence=0.5,
    # 프레임 간 손 추적 신뢰도 임계값 (50% 이상)
    min_tracking_confidence=0.5
)

# 0번 인덱스의 기본 웹캠 디바이스 연결
cap = cv2.VideoCapture(0)

# 가위바위보 게임에서 사용할 무작위 선택 리스트
choices = ["가위", "바위", "보"]

# 컴퓨터가 선택한 낸 값 저장 변수 초기화
computer_choice = None

# 플레이어가 선택한 낸 값 저장 변수 초기화
user_choice = None

# 화면에 표시될 게임 결과 텍스트 초기화
game_result = "Space키를 눌러 가위바위보 시작!"

# 사용자 누적 점수 변수 초기화
score_user = 0

# 컴퓨터 누적 점수 변수 초기화
score_computer = 0

# HandLandmarker 인스턴스를 안전하게 생성하고 자원을 관리하는 with 블록 시작
with HandLandmarker.create_from_options(options) as landmarker:
    # 카메라 장치가 정상적으로 열려 있는 동안 지속 반복
    while cap.isOpened():
        # 카메라로부터 프레임 1개를 읽어옴 (ret: 성공여부, frame: BGR 이미지)
        ret, frame = cap.read()
        
        # 프레임을 읽어오지 못했으면(카메라 연결 끊김 등) 반복문 탈출
        if not ret:
            break

        # 사용자가 거울처럼 보게 하기 위해 좌우 반전 처리
        frame = cv2.flip(frame, 1)
        
        # 화면에 관절을 그리기 위해 프레임의 높이(h)와 너비(w) 추출
        h, w, _ = frame.shape
        
        # OpenCV의 BGR 색상 채널을 MediaPipe용 RGB 색상 채널로 변환
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # NumPy 이미지 배열을 MediaPipe 전용 Image 객체로 전환
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        # VIDEO 모드에 필수적으로 필요한 밀리초(ms) 단위의 타임스탬프 계산
        frame_timestamp_ms = int(time.time() * 1000)
        
        # MediaPipe 모델에 프레임과 타임스탬프를 전달하여 손 관절 추론 수행
        result = landmarker.detect_for_video(mp_image, frame_timestamp_ms)

        # 손 감지가 되지 않았을 때의 기본 상태 텍스트
        current_hand_gesture = "손을 올려주세요"

        # 손 관절 추론 결과가 존재할 경우
        if result.hand_landmarks:
            # 감지된 손 개수만큼 반복 (num_hands=1이므로 1회 실행)
            for hand_landmarks in result.hand_landmarks:
                # 21개 랜드마크의 화면 픽셀 좌표를 담을 리스트
                pixel_coords = []
                
                # 랜드마크 21개 좌표를 순회하며 픽셀 위치 계산
                for lm in hand_landmarks:
                    # 정규화된 0.0~1.0 좌표를 실제 화면 해상도 픽셀 좌표로 변환
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    pixel_coords.append((cx, cy))
                    
                    # 관절 위치에 초록색 원 그리기
                    cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)

                # 연결선 정의 배열(HAND_CONNECTIONS)에 따라 파란색 뼈대 그리기
                for start_idx, end_idx in HAND_CONNECTIONS:
                    cv2.line(frame, pixel_coords[start_idx], pixel_coords[end_idx], (255, 0, 0), 2)

                # 현재 캡처된 손 좌표를 분류 함수로 전달해 가위/바위/보 판단
                current_hand_gesture = classify_rps(hand_landmarks)

        # ======================================================================
        # 4. 게임 화면 UI 및 오버레이 텍스트 출력
        # ======================================================================
        # 상단 누적 스코어판 출력 (흰색)
        cv2.putText(frame, f"Score - You: {score_user} | Com: {score_computer}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        
        # 현재 카메라에 인식 중인 손 모양 텍스트 출력 (노란색)
        cv2.putText(frame, f"Detected Hand: {current_hand_gesture}", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        
        # 승패 및 안내 문구 출력 (초록색)
        cv2.putText(frame, f"Result: {game_result}", (20, 130),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        # 컴퓨터의 선택이 존재할 경우 컴퓨터 선택값 출력 (연보라색)
        if computer_choice:
            cv2.putText(frame, f"Com Choice: {computer_choice}", (20, 170),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 250), 2)

        # 하단 조작법 안내 문구 출력 (회색)
        cv2.putText(frame, "Press 'Space' to Play / 'q' or ESC to Exit", (20, h - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180, 180, 180), 1)

        # 완성된 이미지 프레임을 창에 표시
        cv2.imshow("Rock Paper Scissors Game", frame)

        # 5밀리초 동안 키보드 입력 대기
        key = cv2.waitKey(5) & 0xFF
        
        # 스페이스바(ASCII 32) 입력 시 가위바위보 승패 판정 실행
        if key == 32:
            # 인식된 손 모양이 가위, 바위, 보 중 하나일 때만 게임 진행
            if current_hand_gesture in choices:
                # 사용자의 현재 손 모양 저장
                user_choice = current_hand_gesture
                
                # 컴퓨터의 선택 무작위 추출
                computer_choice = random.choice(choices)

                # 사용자 선택과 컴퓨터 선택이 같은 경우 (무승부)
                if user_choice == computer_choice:
                    game_result = f"Draw! (You: {user_choice} vs Com: {computer_choice})"
                
                # 사용자가 이기는 경우 (가위>보, 바위>가위, 보>바위)
                elif (user_choice == "가위" and computer_choice == "보") or \
                    (user_choice == "바위" and computer_choice == "가위") or \
                    (user_choice == "보" and computer_choice == "바위"):
                    game_result = f"You Win! (You: {user_choice} vs Com: {computer_choice})"
                    score_user += 1 # 사용자 점수 1점 추가
                
                # 컴퓨터가 이기는 경우
                else:
                    game_result = f"You Lose! (You: {user_choice} vs Com: {computer_choice})"
                    score_computer += 1 # 컴퓨터 점수 1점 추가
            
            # 인식 상태가 "알 수 없음"이거나 손을 올리지 않은 경우
            else:
                game_result = "Hand position unclear! Try again."

        # 'q' 키 또는 ESC(ASCII 27) 키 입력 시 게임 루프 종료
        elif key == 27 or key == ord('q'):
            break

# ==============================================================================
# 5. 메인 루프 종료 후 자원 해제
# ==============================================================================
# OpenCv 카메라 비디오 스트림 연결 해제
cap.release()

# 모든 OpenCV 그래픽 창 닫기
cv2.destroyAllWindows()

I0000 00:00:1787709276.395395 2826598 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
W0000 00:00:1787709276.400298 2826601 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787709276.410379 2826602 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


: 

#### PyAutoGUI 

In [1]:
import pyautogui

마우스 제어

In [3]:
# 화면 크기 출력 (예: 1920x1080)
width, height = pyautogui.size()

# 현재 마우스 커서 위치 가져오기
x, y = pyautogui.position()

# 마우스 이동 (X: 100, Y: 200 위치로 1초 동안 이동)
pyautogui.moveTo(100, 200, duration=1.0)

# 클릭 제어
pyautogui.click()                      # 좌클릭
pyautogui.doubleClick()                # 더블 클릭
pyautogui.rightClick()                 # 우클릭
pyautogui.click(x=500, y=500)          # 특정 좌표 클릭

# 스크롤 (양수는 위로, 음수는 아래로)
pyautogui.scroll(-500)

키보드 제안

In [4]:
# 문자열 입력 (한글은 pyperclip 라이브러리와 조합 필요, 영문/숫자 기본 지원)
pyautogui.write('Hello World!', interval=0.1)

# 특정 키 누르기
pyautogui.press('enter')
pyautogui.press('esc')

# 조합키 입력 (예: Ctrl + C, Ctrl + V)
pyautogui.hotkey('ctrl', 'c')
pyautogui.hotkey('ctrl', 'v')

가상마우스

In [ ]:
# OpenCV 라이브러리: 비디오 프레임 캡처, 좌표 변환, 화면 시각화 담당
import cv2

# MediaPipe 라이브러리: 손 관절(21개 랜드마크) 추론을 담당
import mediapipe as mp

# PyAutoGUI 라이브러리: 파이썬으로 진짜 마우스 이동 및 클릭 제어
import pyautogui

# time 라이브러리: MediaPipe Tasks API에 넘겨줄 타임스탬프 생성용
import time

# math 라이브러리: 엄지와 검지 손가락 끝 사이의 유클리드 거리를 계산용
import math

# ==============================================================================
# 1. PyAutoGUI 환경 설정 및 모니터 해상도 구하기
# ==============================================================================
# 마우스 커서가 화면 모서리에 닿았을 때 발생할 수 있는 파이썬 예외 강제 종료 방지
pyautogui.FAILSAFE = False

# 마우스 동작 간 기본 대기 시간을 0.01초로 짧게 설정하여 커서 반응 속도 향상
pyautogui.PAUSE = 0.01

# 현재 사용 중인 모니터 화면의 전체 가로(screen_w), 세로(screen_h) 해상도 획득
screen_w, screen_h = pyautogui.size()

# ==============================================================================
# 2. MediaPipe Tasks API 설정 및 모델 로드
# ==============================================================================
# BaseOptions: .task 모델 파일 경로 지정용 클래스
BaseOptions = mp.tasks.BaseOptions

# HandLandmarker: 실제 손 관절 좌표 추론 클래스
HandLandmarker = mp.tasks.vision.HandLandmarker

# HandLandmarkerOptions: 신뢰도, 추론 모드, 손 개수 등 세부 옵션 지정 클래스
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

# VisionRunningMode: 비디오 프레임 스트림 방식 선택용 클래스
VisionRunningMode = mp.tasks.vision.RunningMode

# 손 관절 마디 연결 정보 정의 (시각화 목적)
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),        # 엄지손가락
    (0, 5), (5, 6), (6, 7), (7, 8),        # 검지손가락
    (5, 9), (9, 10), (10, 11), (11, 12),   # 중지손가락
    (9, 13), (13, 14), (14, 15), (15, 16), # 약지손가락
    (13, 17), (0, 17), (17, 18), (18, 19), (19, 20) # 새끼손가락
]

# HandLandmarker 모델 세부 옵션 지정
options = HandLandmarkerOptions(
    # 로컬 경로에 있는 모델(.task) 지정
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    # 실시간 비디오 스트림 처리 모드로 설정
    running_mode=VisionRunningMode.VIDEO,
    # 최대 인식할 손 개수 (마우스 제어이므로 1개로 제한)
    num_hands=1,
    # 최초 손 감지 신뢰도 임계값
    min_hand_detection_confidence=0.5,
    # 지속 손 추적 신뢰도 임계값
    min_tracking_confidence=0.5
)

# 기본 웹캠 캡처 객체 생성 (0번 카메라)
cap = cv2.VideoCapture(0)

# 이전 프레임의 마우스 커서 위치 기록용 변수 (부드러운 보정 계산용)
prev_x, prev_y = 0, 0

# 마우스 커서의 떨림 현상을 완화하기 위한 보정 비율 (값이 클수록 부드럽지만 약간의 딜레이 발생)
smoothing = 5

# 클릭 상태를 유지/해제하여 프레임마다 연타로 클릭되는 현상을 방지하는 플래그
is_clicked = False

# 카메라 가장자리 여백 영역 설정 (손을 웹캠 끝까지 움직이지 않아도 모니터 구석에 닿도록 함)
margin = 100 

# ==============================================================================
# 3. 실시간 가상 마우스 루프 시작
# ==============================================================================
# Landmarker 인스턴스 자동 자원 관리를 위한 with 블록 시작
with HandLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        # 웹캠 프레임 가져오기 (ret: 성공 여부, frame: BGR 이미지)
        ret, frame = cap.read()
        if not ret:
            break

        # 사용자가 거울 보듯 자연스럽게 조작하도록 좌우 반전
        frame = cv2.flip(frame, 1)
        
        # 현재 카메라 프레임의 높이(h), 너비(w) 추출
        h, w, _ = frame.shape
        
        # OpenCV의 BGR 이미지를 MediaPipe용 RGB 이미지로 변환
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # NumPy 이미지 배열을 MediaPipe 전용 Image 객체로 래핑
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        # VIDEO 모드 필수 요구사항: 밀리초(ms) 단위의 단조 증가 타임스탬프 계산
        frame_timestamp_ms = int(time.time() * 1000)
        
        # MediaPipe 모델로 손 좌표 추론 실행
        result = landmarker.detect_for_video(mp_image, frame_timestamp_ms)

        # 카메라 화면 상에 노란색 가상 조작 영역 Box 표시
        cv2.rectangle(frame, (margin, margin), (w - margin, h - margin), (255, 255, 0), 2)

        # 손 관절 추론 결과가 존재하는 경우
        if result.hand_landmarks:
            for hand_landmarks in result.hand_landmarks:
                # 21개 관절의 픽셀 좌표를 보관할 리스트
                pixel_coords = []
                
                # 랜드마크 정규화 좌표(0.0~1.0)를 해상도에 맞는 픽셀 좌표로 환산하여 시각화
                for lm in hand_landmarks:
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    pixel_coords.append((cx, cy))
                    # 관절 지점에 초록색 점 그리기
                    cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)

                # 손가락 관절 간 파란색 선으로 연결선 그리기
                for start_idx, end_idx in HAND_CONNECTIONS:
                    cv2.line(frame, pixel_coords[start_idx], pixel_coords[end_idx], (255, 0, 0), 2)

                # 마우스 커서 조작 및 클릭 판정에 사용할 엄지 끝(4번)과 검지 끝(8번) 좌표 추출
                thumb_x, thumb_y = pixel_coords[4]
                index_x, index_y = pixel_coords[8]

                # --------------------------------------------------------------
                # 3-1. 마우스 이동 처리 (검지 손가락 끝 좌표 기준)
                # --------------------------------------------------------------
                # 카메라 조작 영역(margin 적용 영역) 좌표를 실제 모니터 해상도 좌표 비율로 선형 매핑
                target_x = (index_x - margin) / (w - 2 * margin) * screen_w
                target_y = (index_y - margin) / (h - 2 * margin) * screen_h

                # 모니터 화면 바깥으로 커서가 벗어나지 않도록 좌표값 상하한 제한
                target_x = max(0, min(screen_w, target_x))
                target_y = max(0, min(screen_h, target_y))

                # 이동 보정 알고리즘: 이전 좌표와 목표 좌표 사이를 보정하여 손떨림으로 인한 마우스 튐 방지
                curr_x = prev_x + (target_x - prev_x) / smoothing
                curr_y = prev_y + (target_y - prev_y) / smoothing

                # PyAutoGUI를 통해 실제 OS 마우스 위치 이동
                pyautogui.moveTo(curr_x, curr_y)
                
                # 다음 프레임 보정을 위한 현재 위치 저장
                prev_x, prev_y = curr_x, curr_y

                # --------------------------------------------------------------
                # 3-2. 마우스 좌클릭 처리 (검지와 엄지 손가락 끝 간 거리 계산)
                # --------------------------------------------------------------
                # 2차원 피타고라스 정리(유클리드 거리)로 두 손가락 끝의 픽셀 거리 계산
                distance = math.hypot(index_x - thumb_x, index_y - thumb_y)
                
                # 엄지와 검지 사이를 연결하는 노란색 직선 그리기 (집게 동작 시각화)
                cv2.line(frame, (thumb_x, thumb_y), (index_x, index_y), (0, 255, 255), 2)

                # 거리가 30픽셀 미만으로 좁혀지면(손가락 맞닿음/집게 동작) 클릭 인식
                if distance < 30:
                    # 클릭 동작 인식 시 검지 끝에 빨간색 동그라미 표시
                    cv2.circle(frame, (index_x, index_y), 10, (0, 0, 255), -1)
                    
                    # 단발성 클릭 처리를 위해 is_clicked 플래그 사용
                    if not is_clicked:
                        pyautogui.click() # 좌클릭 실행
                        is_clicked = True  # 클릭 중 상태로 전환
                else:
                    # 손가락을 떼면 클릭 상태 해제
                    is_clicked = False

        # 가상 마우스 조작 화면 출력
        cv2.imshow("Virtual Mouse", frame)

        # 5밀리초 동안 키 입력 대기 ('q' 키 또는 ESC 입력 시 반복 탈출)
        key = cv2.waitKey(5) & 0xFF
        if key == 27 or key == ord('q'):
            break

# ==============================================================================
# 4. 자원 해제
# ==============================================================================
# 웹캠 장치 연결 해제
cap.release()

# 모든 OpenCV 그래픽 창 닫기
cv2.destroyAllWindows()

I0000 00:00:1787720839.247729 3002924 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 11, prefix = pthread-default
I0000 00:00:1787720839.291300 3002924 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1787720839.304573 3002929 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787720839.325845 3002929 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787720839.378519 3002931 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


: 